# Follow a program through the compiler

This is Notebook 2 for the SC26 tutorial.
We will follow a Qiskit circuit through QC and QCO, simplify it, and simulate the result with DDSIM.

Start with a fresh kernel and run the cells in order.
This notebook uses the same Python 3.15 environment as the rest of the tutorial.

In [ ]:
from IPython.display import Code, display
from qiskit import QuantumCircuit

from mqt.core.mlir import OutputFormat, QCProgram, compile_program, simulate

## Start with a program you can predict

A Hadamard followed by a controlled-$X$ prepares the Bell state $(|00\rangle + |11\rangle)/\sqrt{2}$ from $|00\rangle$.
This time, though, we deliberately use three Hadamards.
Since $H^2 = I$, we should still reach the same state.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.h(0)
bell.h(0)
bell.cx(0, 1)
bell.draw("mpl")

## Read the imported QC program

We begin by importing the Qiskit circuit into QC, one of MQT Core's MLIR dialects.
A **dialect** groups related operations and types.

In [ ]:
qc = QCProgram.from_qiskit(bell)
display(Code(qc.ir, language="mlir"))

Here is how to read the program:

- `func.func @main()` is the entry point, and `return` ends it.
- `arith.constant` provides the indices for qubit access.
- `memref.alloc`, `memref.load`, and `memref.dealloc` manage the register and its qubit references.
- `qc.h` applies $H$, while `qc.ctrl` encloses the controlled-$X$ body ending in `qc.yield`.

The gates reuse qubit references while changing their state, which is **reference semantics**.

## Make quantum data flow explicit with QCO

Now let us convert the program to QCO, where the dependencies between gates become explicit.
With `copy=True`, we keep the QC program available for comparison:

In [ ]:
qco = qc.to_qco(copy=True)
display(Code(qco.ir, language="mlir"))

Each `qco.h` consumes a qubit value and returns the value passed to the next gate.
This is **value semantics**: dependencies are explicit, and each quantum value is used exactly once.
The controlled operation likewise returns the updated control and target values.
The surrounding `qtensor` operations extract qubits from the register and insert the updated values afterward.

## Run one optimization and check its meaning

A **pass** inspects or transforms a program.
For this circuit, we use `simplify-quantum` to cancel adjacent Hadamards on the same qubit.

:::{admonition} Your turn
:class: tip

Before running the next cell, predict the gate count.
:::

In [ ]:
optimized_qco = qco.copy()
optimized_qco.run_pass_pipeline("simplify-quantum")
final_qc = optimized_qco.to_qc(copy=True)

print(f"Gate count before: {qc.num_gates()}")
print(f"Gate count after: {final_qc.num_gates()}")
final_qc.to_qiskit(copy=True).draw("mpl")

We can now simulate the optimized program with DDSIM and inspect the state prepared from $|00\rangle$.

In [ ]:
state = simulate(optimized_qco)
for index, amplitude in enumerate(state):
    print(f"|{index:02b}>: {amplitude.real:+.3f}{amplitude.imag:+.3f}j")

The statevector lists amplitudes in basis order $|q_1q_0\rangle$.
Only $|00\rangle$ and $|11\rangle$ have nonzero amplitudes, so either measurement outcome has probability $1/2$.

## Locate this pass in the full compiler

The full compiler brings these steps together in `compile_program`, from import through optimization to output conversion.
With `output=OutputFormat.QC`, we get the optimized program back in QC:

In [ ]:
final_qc = compile_program(bell, output=OutputFormat.QC)

The final QC program still prepares the Bell state, but the two redundant Hadamards are gone:

In [ ]:
display(Code(final_qc.ir, language="mlir"))

## Checkpoint: Return to the slides

We return to the slides here.
After the break, we will continue with `hardware_compilation.ipynb`.
The remaining experiments are optional.

## Experiment: Which Hadamards cancel?

:::{admonition} Your turn
:class: tip

Predict the effect of zero through four consecutive Hadamards.
Run the cell to check each variant and inspect its remaining gate count.
:::

In [ ]:
for hadamards in range(5):
    circuit = QuantumCircuit(2)
    for _ in range(hadamards):
        circuit.h(0)
    circuit.cx(0, 1)

    qc = QCProgram.from_qiskit(circuit)
    final_qc = compile_program(qc, output=OutputFormat.QC)

    print(f"{hadamards} Hadamards → {final_qc.num_gates()} total gates")

:::{dropdown} Explain the result

An even number of Hadamards acts as the identity, while an odd number acts as one Hadamard.
The controlled-$X$ remains because it still affects other input states.
:::

Let us now replace the three Hadamards with $H$, $Z$, $H$.

:::{admonition} Your turn
:class: tip

Can you still cancel the two Hadamards across $Z$?
Before running the cell, predict the final state.
:::

In [ ]:
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.z(0)
circuit.h(0)
circuit.cx(0, 1)

qc = QCProgram.from_qiskit(circuit)
optimized_qco = compile_program(qc, output=OutputFormat.QCO_OPTIMIZED)

print(simulate(optimized_qco))

:::{dropdown} Explain the result

Since $HZH = X$, the first three gates flip qubit 0 from $|0\rangle$ to $|1\rangle$.
The controlled-$X$ then flips qubit 1, giving $|11\rangle$.
The intervening $Z$ prevents the two Hadamards from cancelling.
:::

:::{admonition} Read further
:class: seealso

See the [Compiler Collection guide](https://mqt.readthedocs.io/projects/core/en/stable/mlir/mqt_compiler_collection.html) for further compiler passes and interfaces.
:::